# Module 05 — Lesson 10
# Pivot Tables

---

> Lesson 07's `groupby()` gave you one long list -- average tip per day, another row for every day. A **pivot table** takes that same idea and lays it out as a grid instead: days down the side, times of day across the top, average tip filling every cell. It's the same summarization you already know how to do, reshaped into the format you'd recognize instantly from Excel. In fact, `pivot_table()` is pandas' answer to Excel's PivotTable feature.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Build a summary grid with `pivot_table(values=, index=, columns=, aggfunc=)`
- Apply multiple aggregation functions in one pivot table
- Add row/column totals with `margins=True`
- Handle missing combinations in the grid with `fill_value`
- Explain why `pivot()` fails on data with repeated rows, and why `pivot_table()` doesn't

In [ ]:
import pandas as pd

tips = pd.read_csv("data/tips.csv")
print(tips.head())

---
## 1. A Basic Pivot Table

`index` becomes the row labels, `columns` becomes the column labels, and `values` is what gets summarized in each cell. By default, `pivot_table()` computes the **mean** of `values` for every `index` × `columns` combination.

In [ ]:
avg_tip_grid = tips.pivot_table(values="tip", index="day", columns="time")
print("Average tip, day (rows) x time (columns):")
print(avg_tip_grid)

Notice `Sat` and `Sun` show `NaN` under `Lunch` -- this restaurant simply has no Saturday or Sunday lunch service in the data. A pivot table doesn't invent data; an empty cell means that combination never occurred.

---
## 2. Choosing the Aggregation: `aggfunc`

The default is `'mean'`, but you can ask for `'sum'`, `'count'`, `'max'`, or any aggregation `groupby()` supports.

In [ ]:
total_tip_grid = tips.pivot_table(values="tip", index="day", columns="time", aggfunc="sum")
print("Total tip $ collected, day x time:")
print(total_tip_grid)
print()

count_grid = tips.pivot_table(values="tip", index="day", columns="time", aggfunc="count")
print("Number of bills, day x time:")
print(count_grid)

---
## 3. Multiple Aggregations at Once

Pass a list to `aggfunc` to compute several statistics per cell in one call -- the result gets an extra column level to hold each one.

In [ ]:
multi_agg = tips.pivot_table(values="tip", index="day", columns="time", aggfunc=["mean", "count"])
print(multi_agg)

---
## 4. Row and Column Totals: `margins=True`

`margins=True` adds an `"All"` row and column showing the aggregation across every day, and across every time -- handy for seeing the grand total alongside the breakdown.

In [ ]:
with_totals = tips.pivot_table(values="tip", index="day", columns="time", aggfunc="sum", margins=True)
print(with_totals)

---
## 5. Filling Missing Combinations: `fill_value`

Rather than leaving genuinely-empty combinations as `NaN`, `fill_value` lets you substitute a specific number -- **only do this when `NaN` and "zero" mean the same thing for your analysis.** Here, 0 tip dollars collected on a day/time that never happened is a reasonable stand-in; don't reach for this automatically everywhere.

In [ ]:
filled = tips.pivot_table(values="tip", index="day", columns="time", aggfunc="sum", fill_value=0)
print(filled)

---
## 6. `pivot()` vs `pivot_table()`

`.pivot()` (no `_table`) reshapes data the same way, but **without aggregating** -- it assumes exactly one row per `index`/`columns` combination. The moment two rows share the same combination, `.pivot()` raises an error, because it has no rule for combining them. `pivot_table()` exists specifically to handle that by aggregating (with `aggfunc`, default `mean`).

In [ ]:
print("tips has MANY rows sharing the same (day, time) combination -- e.g.:")
print(f"  Sat/Dinner alone has {((tips['day'] == 'Sat') & (tips['time'] == 'Dinner')).sum()} rows")
print()

try:
    tips.pivot(index="day", columns="time", values="tip")
except ValueError as e:
    print(f"pivot() error: {e}")
print()
print("pivot_table() handles this fine, because it aggregates duplicates by default:")
print(tips.pivot_table(index="day", columns="time", values="tip").head(2))

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Using pivot() on data with repeated index/column combinations ---

print("Mistake 1 — pivot() assumes ONE row per index/columns pair; real data rarely is:")
try:
    tips.pivot(index="day", columns="time", values="total_bill")
except ValueError as e:
    print(f"  Caught error: {e}")
print()
print("  Correct: use pivot_table(), which aggregates automatically:")
print(tips.pivot_table(index="day", columns="time", values="total_bill").head(2))

In [ ]:
# -- Mistake 2: Forgetting the default aggfunc is 'mean', not 'sum' -------------

print("Mistake 2 — expecting totals but getting averages because aggfunc wasn't set:")
default_grid = tips.pivot_table(values="tip", index="day", columns="time")
print("  Default (mean) -- these are NOT total revenue:")
print(default_grid)
print()
print("  Correct: be explicit when you want totals:")
print(tips.pivot_table(values="tip", index="day", columns="time", aggfunc="sum"))

In [ ]:
# -- Mistake 3: Reading NaN in a pivot table as zero -----------------------------

print("Mistake 3 — NaN means 'no rows existed for this combination', not zero:")
grid = tips.pivot_table(values="tip", index="day", columns="time")
print(grid)
print()
print("  Sat/Lunch is NaN because the restaurant had NO Saturday lunch service --")
print("  that's different from a Saturday lunch shift that made $0 in tips.")
print("  Only use fill_value=0 if that distinction genuinely doesn't matter to you.")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Basic Pivot Table

Build a pivot table with `"day"` as the index, `"smoker"` as the columns, and the mean of `"total_bill"` as the values. Print it.

In [ ]:
# Your code here

### Exercise 2 — `aggfunc='count'`

Using the same `index`/`columns` as Exercise 1, build a pivot table showing the **count** of bills instead of the mean.

In [ ]:
# Your code here

### Exercise 3 — Multiple Aggregations

Build a pivot table with `"time"` as the index, `"day"` as the columns, `"tip"` as the values, and `aggfunc=["mean", "max"]`. Print it.

In [ ]:
# Your code here

### Exercise 4 — `margins=True`

Build a pivot table with `"day"` as the index, `"time"` as the columns, `"tip"` as the values, `aggfunc="sum"`, and `margins=True`. Identify the grand total tip amount from the `"All"` row/column.

In [ ]:
# Your code here

### Exercise 5 — (Challenge) `pivot()` Fails, `pivot_table()` Doesn't

Try `tips.pivot(index="smoker", columns="day", values="total_bill")` inside a `try`/`except ValueError` block, print the caught error, then build the equivalent `pivot_table()` with `fill_value=0` and print that instead.

In [ ]:
# Your code here

---
## 📌 Key Takeaways

- **`pivot_table(values=, index=, columns=, aggfunc=)` reshapes long data into a summary grid** — the same summarization as `groupby()`, laid out as rows × columns instead of one long list. The default `aggfunc` is `'mean'`.

- **`pivot()` only works when each index/columns combination appears once** — real data almost always has repeats, which is exactly why `pivot_table()` (which aggregates by default) is the one you'll reach for in practice.

- **A `NaN` cell in a pivot table means that combination never occurred in the data** — it isn't automatically the same as zero. Only use `fill_value=0` when that distinction genuinely doesn't matter for your analysis.

---
## What's Next?

**Module 05 Project** puts everything from Lessons 01-10 together — loading, filtering, sorting, cleaning, grouping, merging, and pivoting — on a full real-world dataset.